In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import sqlite3
df_2019 = pd.read_csv('/kaggle/input/datasets/duduzilezwane/cyclistic/Divvy_Trips_2019_Q1/Divvy_Trips_2019_Q1.csv')
df_2020 = pd.read_csv('/kaggle/input/datasets/duduzilezwane/cyclistic/Divvy_Trips_2020_Q1/Divvy_Trips_2020_Q1.csv')

In [ ]:
conn = sqlite3.connect('cyclistic.db')

In [ ]:
df_2019.to_sql('divvy_2019_q1', conn, if_exists='replace', index=False)
df_2020.to_sql('divvy_2020_q1', conn, if_exists='replace', index=False)

In [ ]:
##Quick check

df_2019.head()

In [ ]:
##Data cleaning to ensure consistency across datasets 

df_2019.columns

In [ ]:
df_2020.columns

In [ ]:
##Standardizing column names

df_2019 = df_2019.rename(columns={
    'trip_id': 'ride_id',
    'start_time': 'started_at',
    'end_time': 'ended_at',
    'bikeid': 'bike_id'
})

df_2020 = df_2020.rename(columns={
    'trip_id': 'ride_id',
    'start_time': 'started_at',
    'end_time': 'ended_at',
    'bikeid': 'bike_id'
})

In [ ]:
##Converting data columns to timestamps to allow time calculations

df_2019['started_at'] = pd.to_datetime(df_2019['started_at'])
df_2019['ended_at'] = pd.to_datetime(df_2019['ended_at'])

df_2020['started_at'] = pd.to_datetime(df_2020['started_at'])
df_2020['ended_at'] = pd.to_datetime(df_2020['ended_at'])

In [ ]:
##Converting rider types into two groups only

df_2019['rider_type'] = df_2019['usertype'].apply(
    lambda x: 'member' if x == 'Subscriber' else 'casual'
)

df_2020['rider_type'] = df_2020['member_casual'].str.lower().apply(
    lambda x: 'member' if x == 'member' else 'casual'
)

In [ ]:
##cleaner, more readable alternative to previous code

df_2019['rider_type'] = df_2019['usertype'].replace({
    'Subscriber': 'member',
    'Customer': 'casual'
})
df_2020['rider_type'] = df_2020['member_casual'].str.lower()

In [ ]:
##quick verification

df_2020[['member_casual','rider_type']].head()

In [ ]:
##another verification

df_2020['member_casual'].unique()

In [ ]:
##Create calculated field: ride length

df_2019['ride_length'] = (df_2019['ended_at'] - df_2019['started_at']).dt.total_seconds()

df_2020['ride_length'] = (df_2020['ended_at'] - df_2020['started_at']).dt.total_seconds()

In [ ]:
##Create calculated field: day of week

df_2019['day_of_week'] = df_2019['started_at'].dt.day_name()
df_2020['day_of_week'] = df_2020['started_at'].dt.day_name()

In [ ]:
## Removing invalid rides

df_2019 = df_2019[df_2019['ended_at'] >= df_2019['started_at']]
df_2020 = df_2020[df_2020['ended_at'] >= df_2020['started_at']]

In [ ]:
##Combining the two tables into one "all_trips" table

query = """
CREATE TABLE all_trips AS
SELECT
    ride_id,
    started_at,
    ended_at,
    ride_length,
    day_of_week,
    rider_type
FROM divvy_2019_q1

UNION ALL

SELECT
    ride_id,
    started_at,
    ended_at,
    ride_length,
    day_of_week,
    rider_type
FROM divvy_2020_q1
"""
conn.execute(query)

In [ ]:
##quick verification

pd.read_sql_query("SELECT COUNT(*) FROM all_trips", conn)

In [ ]:
##Running core SQL analysis queries to answer the business question

##Total rides by rider type
##This answers, "who rides the bikes more?"

pd.read_sql_query("""
SELECT 
    rider_type,
    COUNT(*) AS total_rides
FROM all_trips
GROUP BY rider_type
""", conn)

## To make it export, visual and chart ready, use
rides_by_type = pd.read_sql_query("""
SELECT 
    rider_type,
    COUNT(*) AS total_rides
FROM all_trips
GROUP BY rider_type
""", conn)

rides_by_type

In [ ]:
##Visual 1

rides_by_type.plot(
    kind='bar',
    x='rider_type',
    y='total_rides',
    title='Total Rides by Rider Type'
)

In [ ]:
####Average ride duration per rider type
##This answers, "who rides longer?"

avg_ride_length = pd.read_sql_query("""
SELECT
    rider_type,
    ROUND(AVG(ride_length)/60,2) AS avg_ride_minutes
FROM all_trips
GROUP BY rider_type
""", conn)

avg_ride_length

In [ ]:
##Visual 2

avg_ride_length.plot(
    kind='bar',
    x='rider_type',
    y='avg_ride_minutes',
    title='Average Ride Duration (Minutes)'
)

In [ ]:
##Rides by day of week
##This answers, "when do members vs casual riders ride?"

rides_by_day = pd.read_sql_query("""
SELECT
    day_of_week,
    rider_type,
    COUNT(*) AS total_rides
FROM all_trips
GROUP BY day_of_week, rider_type
""", conn)

rides_by_day

In [ ]:
##Visual 3

import matplotlib.pyplot as plt

pivot_day = rides_by_day.pivot(index='day_of_week', columns='rider_type', values='total_rides')

pivot_day.plot(kind='bar', title='Rides by Day of Week')

plt.ylabel("Number of Rides")
plt.show()

In [ ]:
##Average ride length by day
##This reveals behavior patterns

avg_ride_by_day = pd.read_sql_query("""
SELECT
    day_of_week,
    rider_type,
    ROUND(AVG(ride_length)/60,2) AS avg_ride_minutes
FROM all_trips
GROUP BY day_of_week, rider_type
""", conn)

avg_ride_by_day

In [ ]:
## Visual 4
pivot_avg = avg_ride_by_day.pivot(index='day_of_week', columns='rider_type', values='avg_ride_minutes')

pivot_avg.plot(kind='bar', title='Average Ride Duration by Day')

plt.ylabel("Average Minutes")
plt.show()

In [ ]:
##Exporting the results
##The aggregated results were exported as CSV files and used to build interactive visualizations in Tableau Public.

rides_by_type.to_csv("rides_by_type.csv", index=False)
avg_ride_length.to_csv("avg_ride_length.csv", index=False)
rides_by_day.to_csv("rides_by_day.csv", index=False)
avg_ride_by_day.to_csv("avg_ride_by_day.csv", index=False)